# 03 · Elo baseline + the evaluation harness

The bar every fancier model must clear. We fit a simple **Elo → outcome** logistic model, then score it on a held-out World Cup with **RPS**, **log-loss**, **Brier**, and a **calibration curve**. Reuse these metrics for every later model.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 40)
PROC = ROOT / 'data' / 'processed'

def load_features():
    fp = PROC / 'match_features.parquet'
    if fp.exists():
        return pd.read_parquet(fp)
    csv = PROC / 'match_features_sample.csv'
    if csv.exists():
        print('Using SAMPLE csv (run 02_build_features.py for the full table).')
        return pd.read_csv(csv, parse_dates=['date'])
    raise FileNotFoundError('No processed data. Run scripts/01_download.py then scripts/02_build_features.py')

df = load_features()
print(df.shape, '|', df['date'].min().date(), '->', df['date'].max().date())

from wcpred import datasets
played = df[df['played']].dropna(subset=['result']).copy()
played['y'] = played['result'].map({'H':0,'D':1,'A':2}).astype(int)


## Metric functions (project-standard)
RPS respects outcome order (H > D > A) and is the standard 1X2 metric; log-loss & Brier complement it.

In [ ]:
from sklearn.metrics import log_loss, brier_score_loss

def rps(probs, y):
    o = np.zeros_like(probs); o[np.arange(len(y)), y] = 1
    cp = np.cumsum(probs, axis=1)[:, :-1]; co = np.cumsum(o, axis=1)[:, :-1]
    return float(((cp - co) ** 2).sum(axis=1).mean())

def report(name, probs, y):
    probs = np.clip(probs, 1e-9, 1)
    print(f'{name:22s}  RPS={rps(probs,y):.4f}  logloss={log_loss(y,probs,labels=[0,1,2]):.4f}')


## Baselines to beat
1. **Naive base rate** — overall H/D/A frequencies, same for every match.
2. **Elo logistic** — multinomial logistic on `elo_diff` (+ neutral flag).

In [ ]:
train, test = datasets.tournament_holdout(played, 2022)  # train < 2022 WC, test = 2022 WC finals
print('train:', len(train), ' test (2022 WC):', len(test))

# 1) Naive base rate from training data
base = train['y'].value_counts(normalize=True).reindex([0,1,2]).values
p_base = np.tile(base, (len(test), 1))
report('naive base-rate', p_base, test['y'].to_numpy())

In [ ]:
from sklearn.linear_model import LogisticRegression
feat = [c for c in ['elo_diff'] if c in train.columns]
Xtr = train[feat].astype(float).fillna(0.0); Xte = test[feat].astype(float).fillna(0.0)
# add neutral indicator
Xtr = Xtr.assign(neutral=train['neutral'].astype(int).values)
Xte = Xte.assign(neutral=test['neutral'].astype(int).values)
elo_lr = LogisticRegression(max_iter=2000).fit(Xtr, train['y'])
p_elo = elo_lr.predict_proba(Xte)
report('elo logistic', p_elo, test['y'].to_numpy())

## Calibration
Are predicted probabilities honest? Bin the predicted P(home win) and compare to the realised rate.

In [ ]:
from sklearn.calibration import calibration_curve
y_home = (test['y'].to_numpy() == 0).astype(int)
frac, mean_pred = calibration_curve(y_home, p_elo[:,0], n_bins=8, strategy='quantile')
plt.plot(mean_pred, frac, 'o-', label='elo logistic'); plt.plot([0,1],[0,1],'k--',alpha=.5)
plt.xlabel('predicted P(home win)'); plt.ylabel('observed'); plt.title('Calibration (2022 WC)'); plt.legend(); plt.show()

## Walk-forward across tournaments
Repeat the Elo baseline for each World Cup so you have a stable RPS to beat with the real model.

In [ ]:
for yr, tr, te in datasets.walk_forward_tournaments(played, (2010,2014,2018,2022)):
    Xtr = tr[['elo_diff']].astype(float).fillna(0).assign(neutral=tr['neutral'].astype(int).values)
    Xte = te[['elo_diff']].astype(float).fillna(0).assign(neutral=te['neutral'].astype(int).values)
    m = LogisticRegression(max_iter=2000).fit(Xtr, tr['y'])
    p = np.clip(m.predict_proba(Xte), 1e-9, 1)
    print(f'WC {yr}:  RPS={rps(p, te["y"].to_numpy()):.4f}  (n={len(te)})')

### Next steps
- Beat these RPS numbers with the **Dixon-Coles goals model** and the **HistGBM/XGBoost** outcome model, then **ensemble** (PLAN.md §5).
- Add **de-vigged bookmaker odds** as the ultimate benchmark when available.
- Once a model beats Elo here, plug it into `wcpred.tournament.simulate_tournament` for the 2026 run.